<a href="https://colab.research.google.com/github/nsharon4/Shablul/blob/main/SuperCart_IL_Session_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request
import re
import os
import html as html_module  # NEW: for decoding &amp; etc.

print("Fetching the Shufersal price index page...")
req = urllib.request.Request(
    "https://prices.shufersal.co.il/FileObject/UpdateCategory?catID=2&storeId=0&sort=Time&sortdir=DESC",
    headers={"User-Agent": "Mozilla/5.0 (SuperCart-IL-research)"}
)
with urllib.request.urlopen(req, timeout=30) as response:
    html_text = response.read().decode("utf-8", errors="replace")
print(f"Got index page: {len(html_text):,} characters")

pattern = r'href="(https://pricesprodpublic\.blob\.core\.windows\.net/pricefull/PriceFull[^"]+\.gz[^"]*)"'
matches = re.findall(pattern, html_text)
print(f"Found {len(matches)} PriceFull file links")

# THE FIX: decode HTML entities (&amp; -> &, etc.) before using URLs
matches = [html_module.unescape(url) for url in matches]

if not matches:
    print("ERROR: No PriceFull links found.")
else:
    success = False
    for i, file_url in enumerate(matches[:3]):
        filename = "shufersal_sample.gz"
        print(f"\nAttempt {i+1}/3...")
        try:
            req2 = urllib.request.Request(
                file_url,
                headers={
                    "User-Agent": "Mozilla/5.0 (SuperCart-IL-research)",
                    "Referer": "https://prices.shufersal.co.il/"
                }
            )
            with urllib.request.urlopen(req2, timeout=60) as response:
                data = response.read()
            with open(filename, "wb") as f:
                f.write(data)
            size_kb = os.path.getsize(filename) / 1024
            print(f"✅ SUCCESS! File size: {size_kb:.1f} KB")
            print(f"Saved as: {filename}")
            success = True
            break
        except urllib.error.HTTPError as e:
            print(f"   Got HTTP {e.code} — trying next file...")
        except Exception as e:
            print(f"   Error: {type(e).__name__}: {e} — trying next file...")

    if not success:
        print("\n❌ Still failing. Paste the output back.")

Fetching the Shufersal price index page...
Got index page: 16,286 characters
Found 20 PriceFull file links

Attempt 1/3...
✅ SUCCESS! File size: 168.7 KB
Saved as: shufersal_sample.gz


In [ ]:
import gzip

# Unzip the file we just downloaded
with gzip.open("shufersal_sample.gz", "rb") as f:
    xml_bytes = f.read()

xml_text = xml_bytes.decode("utf-8", errors="replace")

print(f"📄 Decompressed XML size: {len(xml_text):,} characters")
print(f"📄 Roughly {xml_text.count('<Item>'):,} products in this file\n")
print("=" * 70)
print("FIRST 3,000 CHARACTERS OF THE RAW XML:")
print("=" * 70)
print(xml_text[:3000])

📄 Decompressed XML size: 2,581,021 characters
📄 Roughly 3,214 products in this file

FIRST 3,000 CHARACTERS OF THE RAW XML:
﻿<Root>
  <ChainID>7290027600007</ChainID>
  <SubChainID>001</SubChainID>
  <StoreID>357</StoreID>
  <BikoretNo>5</BikoretNo>
  <Items>
    <Item>
      <PriceUpdateTime>2025-04-29T09:21:00</PriceUpdateTime>
      <ItemCode>11210000094</ItemCode>
      <LastSaleDateTime>2026-04-17T12:59:02</LastSaleDateTime>
      <ItemType>1</ItemType>
      <ItemName>רוטב טבסקו 60 מ"ל</ItemName>
      <ManufactureName>ניצן</ManufactureName>
      <ManufactureCountry>US</ManufactureCountry>
      <ManufactureItemDescription>רוטב טבסקו 60 מ"ל</ManufactureItemDescription>
      <UnitQty>מיליליטר</UnitQty>
      <Quantity>0.60</Quantity>
      <UnitOfMeasure>100 מיליליטר</UnitOfMeasure>
      <bIsWeighted>0</bIsWeighted>
      <QtyInPackage>0</QtyInPackage>
      <ItemPrice>12.90</ItemPrice>
      <UnitOfMeasurePrice>21.50</UnitOfMeasurePrice>
      <AllowDiscount>1</AllowDiscount>


In [ ]:
import xml.etree.ElementTree as ET
import gzip

# Parse the XML properly
with gzip.open("shufersal_sample.gz", "rb") as f:
    xml_bytes = f.read()

root = ET.fromstring(xml_bytes)

chain_id = root.findtext("ChainID")
store_id = root.findtext("StoreID")
items = root.find("Items")

print(f"🏪 Chain ID: {chain_id} (Shufersal)")
print(f"🏬 Store ID: {store_id}")
print(f"📦 Total products in this file: {len(items):,}")
print()

# Extract every product into a clean list of dictionaries
products = []
for item in items.findall("Item"):
    products.append({
        "barcode":         item.findtext("ItemCode"),
        "name_he":         item.findtext("ItemName"),
        "manufacturer":    item.findtext("ManufactureName"),
        "country":         item.findtext("ManufactureCountry"),
        "price_nis":       float(item.findtext("ItemPrice") or 0),
        "unit_price_nis":  float(item.findtext("UnitOfMeasurePrice") or 0),
        "unit_type":       item.findtext("UnitQty"),
        "quantity":        item.findtext("Quantity"),
        "last_updated":    item.findtext("PriceUpdateTime"),
    })

print(f"✅ Extracted {len(products):,} products into clean Python objects")
print()

# Show the first 5, nicely formatted
print("=" * 70)
print("FIRST 5 PRODUCTS")
print("=" * 70)
for i, p in enumerate(products[:5], 1):
    print(f"\n#{i}")
    print(f"  Hebrew name:  {p['name_he']}")
    print(f"  Manufacturer: {p['manufacturer']}  ({p['country']})")
    print(f"  Barcode:      {p['barcode']}")
    print(f"  Price:        ₪{p['price_nis']:.2f}")
    print(f"  Unit price:   ₪{p['unit_price_nis']:.2f} per {p['unit_type']}")

🏪 Chain ID: 7290027600007 (Shufersal)
🏬 Store ID: 357
📦 Total products in this file: 3,214

✅ Extracted 3,214 products into clean Python objects

FIRST 5 PRODUCTS

#1
  Hebrew name:  רוטב טבסקו 60 מ"ל
  Manufacturer: ניצן  (US)
  Barcode:      11210000094
  Price:        ₪12.90
  Unit price:   ₪21.50 per מיליליטר

#2
  Hebrew name:  רוטב טבסקו ירוק 60 מ"ל
  Manufacturer: ניצן  (US)
  Barcode:      11210009387
  Price:        ₪12.90
  Unit price:   ₪21.50 per מיליליטר

#3
  Hebrew name:  טבסקו סרירצה 566 מ"ל
  Manufacturer: מקאילהייני  (US)
  Barcode:      11210698055
  Price:        ₪29.90
  Unit price:   ₪5.28 per מיליליטר

#4
  Hebrew name:  חטיף שיבולת ושוקולד5
  Manufacturer: גנרל מילס סאן אדריאן  (ES)
  Barcode:      16000423534
  Price:        ₪21.90
  Unit price:   ₪10.43 per גרם

#5
  Hebrew name:  חטיף נייטשר וואלי דבש5יח
  Manufacturer: גנרל מילס סאן אדריאן  (ES)
  Barcode:      16000548404
  Price:        ₪21.90
  Unit price:   ₪10.43 per גרם


In [ ]:
import csv

# Write all 3,214 products to a CSV file
csv_path = "shufersal_store_357_products.csv"

with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "barcode", "name_he", "manufacturer", "country",
        "price_nis", "unit_price_nis", "unit_type", "quantity", "last_updated"
    ])
    writer.writeheader()
    writer.writerows(products)

print(f"✅ Saved {len(products):,} products to '{csv_path}'")

# Quick stats so you can see what you've got
import statistics
prices = [p["price_nis"] for p in products if p["price_nis"] > 0]
print(f"\n📊 Price statistics across all {len(prices):,} products:")
print(f"   Cheapest:  ₪{min(prices):.2f}")
print(f"   Most expensive: ₪{max(prices):.2f}")
print(f"   Median:    ₪{statistics.median(prices):.2f}")
print(f"   Mean:      ₪{statistics.mean(prices):.2f}")

# Top 10 most expensive
print(f"\n💰 Top 5 most expensive items in this branch:")
for p in sorted(products, key=lambda x: x["price_nis"], reverse=True)[:5]:
    print(f"   ₪{p['price_nis']:>8.2f}  {p['name_he']}")

# Top 5 cheapest (excluding items under ₪1, which are often bag/deposit fees)
real = [p for p in products if p["price_nis"] >= 1]
print(f"\n🪙 5 cheapest real items:")
for p in sorted(real, key=lambda x: x["price_nis"])[:5]:
    print(f"   ₪{p['price_nis']:>8.2f}  {p['name_he']}")

# Country of origin breakdown
from collections import Counter
countries = Counter(p["country"] for p in products if p["country"])
print(f"\n🌍 Top 5 countries of origin:")
for country, count in countries.most_common(5):
    pct = 100 * count / len(products)
    print(f"   {country or '(unknown)':<10} {count:>4} products  ({pct:.1f}%)")

✅ Saved 3,214 products to 'shufersal_store_357_products.csv'

📊 Price statistics across all 3,214 products:
   Cheapest:  ₪2.09
   Most expensive: ₪505.00
   Median:    ₪14.90
   Mean:      ₪18.17

💰 Top 5 most expensive items in this branch:
   ₪  505.00  ווג כחול פאקט
   ₪  505.00  ווג לילך פאקט
   ₪  205.90  סכיני גילוח פרוגלייד 8יח
   ₪  179.60  גלנמורנג'י 12 שנה 700 מ"
   ₪  155.00  ג'ילט פרוגלייד 10 יחידות

🪙 5 cheapest real items:
   ₪    2.09  מלח בישול גס שופרסל 1 קג
   ₪    2.09  מלח שולחן דק שופרסל 1 קג
   ₪    2.10  שמרים טריים בקוביה
   ₪    2.20  נר נשמה 24 שעות יחידה
   ₪    2.30  מיני מילקי ט.שוקולד90מל

🌍 Top 5 countries of origin:
   IL         1852 products  (57.6%)
   IT          239 products  (7.4%)
   CN          122 products  (3.8%)
   PL          105 products  (3.3%)
   DE           95 products  (3.0%)
